In [ ]:
# Cell 1: Title and Google Drive mount
from IPython.display import Markdown, display
display(Markdown('# Frozen VAE PCA Workflow (Varimax-first)'))
display(Markdown(
    'This notebook keeps the frozen VAE and PCA assets fixed, uses Varimax as the default control space, '
    'locks one Intensity axis, and then discovers additional metric-aware control directions.'
))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


In [ ]:
# Cell 2: Repository, frozen manifest, imports, and output folders
display(Markdown('### Cell 2: Repository, frozen manifest, imports, and output folders'))
from pathlib import Path
import json, os, pickle, shutil, subprocess, sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_URL = 'https://github.com/cindy-77jiayi/thesis_hapticAE.git'
PROJECT_ROOT = Path('/content/thesis_hapticAE')
BRANCH = None
FORCE_CLEAN_CLONE = False
INSTALL_REQUIREMENTS = True

if FORCE_CLEAN_CLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not (PROJECT_ROOT / '.git').exists():
    clone_cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    if BRANCH:
        clone_cmd = ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all'], check=False)
    if BRANCH:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=False)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=False)

print('Current branch:')
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], check=True)

requirements_path = PROJECT_ROOT / 'requirements.txt'
if INSTALL_REQUIREMENTS and requirements_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

import importlib
for mod in list(sys.modules):
    if mod == 'src' or mod.startswith('src.'):
        del sys.modules[mod]
importlib.invalidate_caches()

DRIVE_ROOT = Path('/content/drive/MyDrive')
FROZEN_ROOT = DRIVE_ROOT / 'thesis' / 'frozen_model_outputs'
FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'latest_frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'vae_balanced' / 'frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Frozen manifest not found: {FROZEN_MANIFEST_PATH}')

FROZEN_MANIFEST = json.loads(FROZEN_MANIFEST_PATH.read_text(encoding='utf-8'))
FROZEN_RUN_DIR = Path(FROZEN_MANIFEST.get('frozen_run_dir', FROZEN_MANIFEST_PATH.parent))
CONFIG_PATH = Path(FROZEN_MANIFEST['config_path'])
CHECKPOINT_PATH = Path(FROZEN_MANIFEST['checkpoint_path'])
FROZEN_PCA_DIR = Path(FROZEN_MANIFEST.get('pca_dir') or (FROZEN_RUN_DIR / 'pca'))
FROZEN_CONTROLS_DIR = Path(FROZEN_MANIFEST.get('controls_dir') or (FROZEN_RUN_DIR / 'controls'))
FROZEN_LATENT_PATH = Path(FROZEN_MANIFEST.get('latent_path') or (FROZEN_PCA_DIR / 'Z_mu.npy'))
SAMPLE_IDS_PATH = Path(FROZEN_MANIFEST.get('sample_ids_path') or (FROZEN_PCA_DIR / 'sample_ids.json'))


def _looks_like_dataset_root(path: Path) -> bool:
    return path.exists() and any((path / sub).exists() for sub in ('expertvoted', 'uservoted'))


def _discover_dataset_root() -> Path | None:
    env_dir = os.environ.get('HAPTIC_DATA_DIR')
    candidates = []
    if env_dir:
        candidates.append(Path(env_dir))
    candidates.extend([
        Path('/content/hapticgen-dataset'),
        DRIVE_ROOT / 'hapticgen-dataset',
        DRIVE_ROOT / 'thesis' / 'hapticgen-dataset',
        PROJECT_ROOT / 'hapticgen-dataset',
    ])
    for candidate in candidates:
        if _looks_like_dataset_root(candidate):
            return candidate
    return None


DATA_DIR = _discover_dataset_root()

ANALYSIS_NAME = 'frozen_vae_pca_workflow'
OUTPUT_DIR = FROZEN_RUN_DIR / 'analysis' / ANALYSIS_NAME
LATENT_DIR = OUTPUT_DIR / 'latent_cache'
PCA_DIR = OUTPUT_DIR / 'pca_cache'
VARIMAX_DIR = OUTPUT_DIR / 'varimax_cache'
SWEEP_DIR = OUTPUT_DIR / 'sweeps'
SUMMARY_DIR = OUTPUT_DIR / 'summary'
FAMILY_DIR = OUTPUT_DIR / 'family_directions'
COMPARISON_DIR = OUTPUT_DIR / 'comparison'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for d in [OUTPUT_DIR, LATENT_DIR, PCA_DIR, VARIMAX_DIR, SWEEP_DIR, SUMMARY_DIR, FAMILY_DIR, COMPARISON_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.control_spec import compute_control_ranges
from src.pipelines.pca_control import (
    control_to_latent,
    fit_pca_pipeline,
    fit_rotated_pca_pipeline,
    get_explained_variance_ratio,
    plot_sweep,
    sweep_axis,
    sweep_direction,
)
from src.eval.pc_validation import (
    DEFAULT_CAUTION_METRICS,
    DEFAULT_METRIC_FAMILY_MAP,
    ENGINEERING_CONTROL_FAMILIES,
    build_control_samples_from_sweep_payload,
    build_family_profiles,
    choose_anchor_indices,
    compute_axis_alignment,
    discover_metric_family_directions,
    summarize_candidate_axes,
    summarize_multi_anchor_metrics,
)
from src.semantic.pc_semantics import build_semantic_control_table, load_semantic_schema

print('Frozen manifest:', FROZEN_MANIFEST_PATH)
print('Frozen run dir:', FROZEN_RUN_DIR)
print('CONFIG_PATH:', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())
print('CHECKPOINT_PATH:', CHECKPOINT_PATH, 'exists=', CHECKPOINT_PATH.exists())
print('FROZEN_PCA_DIR:', FROZEN_PCA_DIR, 'exists=', FROZEN_PCA_DIR.exists())
print('FROZEN_LATENT_PATH:', FROZEN_LATENT_PATH, 'exists=', FROZEN_LATENT_PATH.exists())
print('SAMPLE_IDS_PATH:', SAMPLE_IDS_PATH, 'exists=', SAMPLE_IDS_PATH.exists())
print('DATA_DIR:', DATA_DIR if DATA_DIR is not None else 'not found (ok if frozen latent/sample_ids exist)')
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# Cell 3: Load config, sample ids, and VAE checkpoint
display(Markdown('### Cell 3: Load config, sample ids, and VAE checkpoint'))

config = load_config(str(CONFIG_PATH))
set_seed(config.get('seed', 42))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T = int(config['data']['T'])
sr = int(config['data']['sr'])

data = None
sample_ids = []
if SAMPLE_IDS_PATH.exists():
    sample_ids = json.loads(SAMPLE_IDS_PATH.read_text(encoding='utf-8'))
    print('Loaded frozen sample ids:', len(sample_ids))
else:
    print('Frozen sample_ids.json not found; sample ids will be rebuilt only if dataset access is needed.')

model = build_model(config, DEVICE)
load_checkpoint(model, str(CHECKPOINT_PATH), DEVICE)
model.eval()

print('Device:', DEVICE)
print('Signal length:', T)
print('Sample rate:', sr)


In [ ]:
# Cell 4: Load frozen latent cache or optionally re-extract mu
display(Markdown('### Cell 4: Load frozen latent cache or optionally re-extract mu'))

LATENT_PATH = LATENT_DIR / 'Z_mu.npy'
if LATENT_PATH.exists():
    Z = np.load(LATENT_PATH)
    print('Loaded analysis latent cache:', LATENT_PATH, Z.shape)
elif FROZEN_LATENT_PATH.exists():
    Z = np.load(FROZEN_LATENT_PATH)
    np.save(LATENT_PATH, Z.astype(np.float32))
    print('Copied frozen latent cache into analysis dir:', LATENT_PATH, Z.shape)
else:
    if DATA_DIR is None:
        raise FileNotFoundError(
            'No dataset root was found and no frozen latent cache is available. '
            'Either mount the dataset and set HAPTIC_DATA_DIR, or regenerate the frozen latent cache first.'
        )
    data = build_dataloaders(config, str(DATA_DIR), batch_size=64, full_dataset=True)
    sample_ids = [Path(p).name for p in data['wav_files']]
    Z = extract_latent_vectors(model, data['all_loader'], DEVICE)
    np.save(LATENT_PATH, Z.astype(np.float32))
    print('Extracted and saved latent cache:', LATENT_PATH, Z.shape)

if not sample_ids:
    if SAMPLE_IDS_PATH.exists():
        sample_ids = json.loads(SAMPLE_IDS_PATH.read_text(encoding='utf-8'))
    else:
        sample_ids = [f'sample_{i:05d}' for i in range(Z.shape[0])]
        print('Using synthetic sample ids because sample_ids.json is unavailable.')

assert len(sample_ids) == Z.shape[0], f'sample_ids length {len(sample_ids)} does not match latent rows {Z.shape[0]}'
print('Latent rows / sample ids:', Z.shape[0], len(sample_ids))


In [ ]:
# Cell 5: Load frozen PCA artifacts or optionally refit baseline PCA
display(Markdown('### Cell 5: Load frozen PCA artifacts or optionally refit baseline PCA'))

PCA_PIPE_PATH = FROZEN_PCA_DIR / 'pca_pipe.pkl'
PCA_SCORE_PATH = FROZEN_PCA_DIR / 'Z_pca.npy'

if PCA_PIPE_PATH.exists() and PCA_SCORE_PATH.exists():
    with open(PCA_PIPE_PATH, 'rb') as f:
        baseline_pipe = pickle.load(f)
    Z_pca = np.load(PCA_SCORE_PATH)
    print('Loaded frozen PCA artifacts')
else:
    baseline_pipe, Z_pca = fit_pca_pipeline(Z, n_components=8, save_dir=str(PCA_DIR))
    print('Refit baseline PCA because frozen artifacts were missing')

baseline_evr = get_explained_variance_ratio(baseline_pipe)
print('Baseline PCA shape:', Z_pca.shape)
print('Baseline cumulative EVR:', float(np.sum(baseline_evr)))


In [ ]:
# Cell 6: Fit or load rotated Varimax control space
display(Markdown('### Cell 6: Fit or load rotated Varimax control space'))

VARIMAX_PIPE_PATH = VARIMAX_DIR / 'varimax_pipe.pkl'
VARIMAX_SCORE_PATH = VARIMAX_DIR / 'Z_varimax.npy'

if VARIMAX_PIPE_PATH.exists() and VARIMAX_SCORE_PATH.exists():
    with open(VARIMAX_PIPE_PATH, 'rb') as f:
        varimax_pipe = pickle.load(f)
    Z_varimax = np.load(VARIMAX_SCORE_PATH)
    print('Loaded cached Varimax artifacts')
else:
    varimax_pipe, Z_varimax = fit_rotated_pca_pipeline(
        baseline_pipe,
        Z_pca=Z_pca,
        save_dir=str(VARIMAX_DIR),
        prefix='varimax',
    )
    print('Fitted and saved Varimax artifacts')

varimax_evr = get_explained_variance_ratio(varimax_pipe)
print('Varimax shape:', Z_varimax.shape)
print('Varimax cumulative EVR:', float(np.sum(varimax_evr)))

pd.DataFrame({
    'axis': [f'PC{i+1}' for i in range(len(varimax_evr))],
    'explained_variance_ratio': np.round(varimax_evr, 6),
})


In [ ]:
# Cell 7: Varimax sweep setup, engineering taxonomy, and shared helpers
display(Markdown('### Cell 7: Varimax sweep setup, engineering taxonomy, and shared helpers'))

ENGINEERING_FAMILIES = list(ENGINEERING_CONTROL_FAMILIES)
METRIC_FAMILY_MAP = dict(DEFAULT_METRIC_FAMILY_MAP)
CAUTION_LABEL_METRICS = set(DEFAULT_CAUTION_METRICS)

EXCLUSIVE_INTENSITY_FAMILY = 'Amplitude / Intensity (Voltage)'
INTENSITY_AXIS_CANDIDATES = ['PC1', 'PC2']

N_ANCHORS = 5
N_SWEEP_STEPS = 21
SINGLE_AXIS_STEPS = 9

anchor_indices = choose_anchor_indices(Z_varimax, n_anchors=N_ANCHORS, seed=config.get('seed', 42))
anchor_table = pd.DataFrame([
    {'anchor_id': i, 'row_index': int(idx), 'sample_id': sample_ids[int(idx)]}
    for i, idx in enumerate(anchor_indices)
])
varimax_anchor_refs = Z_varimax[anchor_indices].astype(np.float32)
baseline_anchor_refs = Z_pca[anchor_indices].astype(np.float32)

varimax_ranges = compute_control_ranges(Z_varimax)
baseline_ranges = compute_control_ranges(Z_pca)

family_taxonomy_table = pd.DataFrame([
    {'control_family': family, 'metrics': ', '.join(sorted([m for m, fam in METRIC_FAMILY_MAP.items() if fam == family]))}
    for family in ENGINEERING_FAMILIES
])

def _axis_range(axis, ranges):
    match = [r for r in ranges if int(r['axis']) == int(axis)]
    if not match:
        raise ValueError(f'No sweep range found for axis {axis}')
    return (float(match[0]['p5']), float(match[0]['p95']))

def _direction_range(score_matrix, direction, percentiles=(5.0, 95.0)):
    direction = np.asarray(direction, dtype=np.float32).reshape(-1)
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    projection = np.asarray(score_matrix, dtype=np.float32) @ direction
    lo, hi = np.percentile(projection, percentiles)
    return (float(lo), float(hi))

def run_multi_anchor_axis_sweeps(pipe, anchor_refs, ranges, sweep_name, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    payload_path = out_dir / f'{sweep_name}_multi_anchor_sweeps.json'

    if payload_path.exists():
        payload = json.loads(payload_path.read_text(encoding='utf-8'))
        print('Loaded cached sweep payload:', payload_path)
        return payload

    payload = {
        'method': sweep_name,
        'n_steps': int(N_SWEEP_STEPS),
        'sweeps': {},
    }

    for axis in range(anchor_refs.shape[1]):
        pc = f'PC{axis + 1}'
        lo, hi = _axis_range(axis, ranges)
        payload['sweeps'][pc] = []
        for anchor_id, reference in enumerate(anchor_refs):
            sweep = sweep_axis(
                pipe,
                model,
                DEVICE,
                axis=axis,
                sweep_range=(lo, hi),
                n_steps=N_SWEEP_STEPS,
                T=T,
                sr=sr,
                reference=reference,
                with_metrics=True,
            )
            payload['sweeps'][pc].append({
                'anchor_id': int(anchor_id),
                'row_index': int(anchor_indices[anchor_id]),
                'sample_id': sample_ids[int(anchor_indices[anchor_id])],
                'values': [float(v) for v in sweep['values']],
                'metrics': [{k: float(v) for k, v in item.items()} for item in sweep['metrics']],
            })

    payload_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('Saved sweep payload:', payload_path)
    return payload

def summarize_sweep_payload(sweep_payload):
    return {
        pc: summarize_multi_anchor_metrics(items)
        for pc, items in sweep_payload['sweeps'].items()
    }

def build_method_overview(method_name, summary_payload):
    df = pd.DataFrame(summary_payload['rows'])
    return {
        'method': method_name,
        'n_use': int((df['use_recommendation'] == 'use').sum()),
        'n_review': int((df['use_recommendation'] == 'review').sum()),
        'mean_overall_score': round(float(df['overall_score'].mean()), 6),
        'mean_selectivity': round(float(df['selectivity'].mean()), 6),
        'mean_family_selectivity': round(float(df['family_selectivity'].mean()), 6),
        'mean_cross_anchor_consistency': round(float(df['cross_anchor_consistency'].mean()), 6),
    }

display(anchor_table)
display(family_taxonomy_table)


In [ ]:
# Cell 8: Varimax multi-anchor sweeps
display(Markdown('### Cell 8: Varimax multi-anchor sweeps'))

VARIMAX_SWEEP_SUBDIR = SWEEP_DIR / 'varimax'
varimax_sweep_payload = run_multi_anchor_axis_sweeps(
    varimax_pipe,
    varimax_anchor_refs,
    varimax_ranges,
    sweep_name='varimax',
    out_dir=VARIMAX_SWEEP_SUBDIR,
)
varimax_pc_summaries = summarize_sweep_payload(varimax_sweep_payload)

pd.DataFrame(varimax_ranges)


In [ ]:
# Cell 9: Canonical semantic control definitions
display(Markdown('### Cell 9: Canonical semantic control definitions'))

display(Markdown('''The semantic source of truth is fixed for the first five PCs:

- `PC1 = frequency`
- `PC2 = intensity` (inverted in PCA space)
- `PC3 = envelope_modulation`
- `PC4 = temporal_grouping`
- `PC5 = sharpness`

Metrics in later tables are supporting evidence only; they must not reinterpret these canonical semantics.

The PCA ranges below are calibrated from `varimax_multi_anchor_sweeps.json`.'''))

SEMANTIC_SCHEMA = load_semantic_schema()
SEMANTIC_CONTROL_TABLE = pd.DataFrame(build_semantic_control_table(SEMANTIC_SCHEMA))
CANONICAL_CONTROL_TABLE = SEMANTIC_CONTROL_TABLE[SEMANTIC_CONTROL_TABLE['status'] == 'canonical'].copy()
UNRESOLVED_CONTROL_TABLE = SEMANTIC_CONTROL_TABLE[SEMANTIC_CONTROL_TABLE['status'] == 'unresolved'].copy()

CONTROL_SPEC_BY_PC = {
    spec['pc_name']: spec
    for spec in SEMANTIC_SCHEMA['semantic_controls']
}
CONTROL_SPEC_BY_PC.update({
    spec['pc_name']: spec
    for spec in SEMANTIC_SCHEMA.get('unresolved_pcs', [])
})

FORCED_PC_FAMILY_LOCKS = {
    'PC1': CONTROL_SPEC_BY_PC['PC1']['canonical_name'],
    'PC2': CONTROL_SPEC_BY_PC['PC2']['canonical_name'],
}
LOCKED_PC_ORDER = ['PC1', 'PC2']
LOCKED_FAMILIES = set(FORCED_PC_FAMILY_LOCKS.values())

varimax_candidate_summary = summarize_candidate_axes(
    varimax_pc_summaries,
    explained_variance_ratio=varimax_evr,
    metric_family_map=METRIC_FAMILY_MAP,
    caution_metrics=CAUTION_LABEL_METRICS,
)

varimax_candidate_df = pd.DataFrame(varimax_candidate_summary['rows'])
varimax_candidate_df['recommendation_rank'] = varimax_candidate_df['use_recommendation'].map({'use': 2, 'review': 1, 'disable': 0})
varimax_candidate_df = varimax_candidate_df.sort_values(
    ['recommendation_rank', 'overall_score'],
    ascending=[False, False],
).reset_index(drop=True)

semantic_schema_copy_path = SUMMARY_DIR / 'semantic_control_schema.json'
semantic_schema_table_path = SUMMARY_DIR / 'semantic_control_table.csv'
varimax_candidate_summary_path = SUMMARY_DIR / 'varimax_candidate_summary.csv'
varimax_candidate_json_path = SUMMARY_DIR / 'varimax_candidate_summary.json'

semantic_schema_copy_path.write_text(json.dumps(SEMANTIC_SCHEMA, indent=2), encoding='utf-8')
SEMANTIC_CONTROL_TABLE.to_csv(semantic_schema_table_path, index=False)
varimax_candidate_df.to_csv(varimax_candidate_summary_path, index=False)
varimax_candidate_json_path.write_text(json.dumps(varimax_candidate_summary, indent=2), encoding='utf-8')

locked_support_rows = []
for pc_name in LOCKED_PC_ORDER:
    support = varimax_candidate_df[varimax_candidate_df['pc'] == pc_name].iloc[0].to_dict()
    spec = CONTROL_SPEC_BY_PC[pc_name]
    locked_support_rows.append({
        'pc': pc_name,
        'canonical_name': spec['canonical_name'],
        'control_name': spec['control_name'],
        'supporting_top_metric': support['top_metric'],
        'supporting_overall_score': support['overall_score'],
        'supporting_confidence': support['confidence'],
        'supporting_use_recommendation': support['use_recommendation'],
    })
locked_support_df = pd.DataFrame(locked_support_rows)

print('Canonical semantic control table:')
display(CANONICAL_CONTROL_TABLE)
print('Unresolved placeholders (PC6-PC8):')
display(UNRESOLVED_CONTROL_TABLE)
print('Supporting metrics for locked controls (evidence only):')
display(locked_support_df)

In [ ]:
# Cell 10: Semantic control overview and machine-readable outputs
display(Markdown('### Cell 10: Semantic control overview'))

display(Markdown('''This table is the semantic-first control view used for downstream mapping and future LLM integration.

- `PC1-PC5` are fixed semantic controls.
- `PC6-PC8` remain unresolved placeholders.
- `pc_range` values are calibrated from `varimax_multi_anchor_sweeps.json`.
- Supporting metrics are shown only as evidence, not as naming logic.'''))

semantic_overview_rows = []
for _, row in SEMANTIC_CONTROL_TABLE.iterrows():
    pc_name = row['control_id']
    candidate_match = varimax_candidate_df[varimax_candidate_df['pc'] == pc_name]
    support = candidate_match.iloc[0].to_dict() if not candidate_match.empty else {}
    semantic_overview_rows.append({
        'control_id': row['control_id'],
        'canonical_name': row['canonical_name'],
        'pc_index': row['pc_index'],
        'control_name': row['control_name'],
        'description': row['description'],
        'low_end_interpretation': row['low_end_interpretation'],
        'high_end_interpretation': row['high_end_interpretation'],
        'semantic_range': row['semantic_range'],
        'pc_range': row['pc_range'],
        'range_source': row['range_source'],
        'semantic_space_direction': row['semantic_space_direction'],
        'pca_space_direction': row['pca_space_direction'],
        'inversion': bool(row['inversion']),
        'status': row['status'],
        'supporting_top_metric': support.get('top_metric'),
        'supporting_overall_score': support.get('overall_score'),
        'supporting_confidence': support.get('confidence'),
        'supporting_use_recommendation': support.get('use_recommendation'),
        'notes': row['notes'],
    })
semantic_control_overview_df = pd.DataFrame(semantic_overview_rows)
semantic_control_overview_df.to_csv(SUMMARY_DIR / 'semantic_control_overview.csv', index=False)
(SUMMARY_DIR / 'semantic_control_overview.json').write_text(
    semantic_control_overview_df.to_json(orient='records', indent=2),
    encoding='utf-8',
)

semantic_control_table_df = semantic_control_overview_df[[
    'control_id',
    'canonical_name',
    'pc_index',
    'control_name',
    'description',
    'low_end_interpretation',
    'high_end_interpretation',
    'semantic_range',
    'pc_range',
    'range_source',
    'semantic_space_direction',
    'pca_space_direction',
    'inversion',
    'status',
    'notes',
]].copy()
semantic_control_table_df.to_csv(SUMMARY_DIR / 'semantic_control_table.csv', index=False)

print('Semantic-first control overview:')
display(semantic_control_overview_df)

In [ ]:
# Cell 11: Canonical control registry for PC3-PC8 under locked PC1/PC2
display(Markdown('### Cell 11: Canonical control registry for `PC3-PC8` under locked `PC1/PC2`'))

LOCKED_CONTROL_DIR = SUMMARY_DIR / 'locked_controls'
LOCKED_CONTROL_DIR.mkdir(parents=True, exist_ok=True)

locked_pc_controls = {
    'PC1': {'pc': 'PC1', 'axis': 0, 'canonical_name': CONTROL_SPEC_BY_PC['PC1']['canonical_name'], 'control_name': CONTROL_SPEC_BY_PC['PC1']['control_name'], 'mode': 'locked_pc'},
    'PC2': {'pc': 'PC2', 'axis': 1, 'canonical_name': CONTROL_SPEC_BY_PC['PC2']['canonical_name'], 'control_name': CONTROL_SPEC_BY_PC['PC2']['control_name'], 'mode': 'locked_pc'},
}
locked_pc_controls_table = pd.DataFrame([
    {
        'pc': 'PC1',
        'canonical_name': CONTROL_SPEC_BY_PC['PC1']['canonical_name'],
        'control_name': CONTROL_SPEC_BY_PC['PC1']['control_name'],
        'control_description': CONTROL_SPEC_BY_PC['PC1']['description'],
        'mode': 'locked_pc',
        'top_metric': varimax_candidate_df[varimax_candidate_df['pc'] == 'PC1'].iloc[0]['top_metric'],
    },
    {
        'pc': 'PC2',
        'canonical_name': CONTROL_SPEC_BY_PC['PC2']['canonical_name'],
        'control_name': CONTROL_SPEC_BY_PC['PC2']['control_name'],
        'control_description': CONTROL_SPEC_BY_PC['PC2']['description'],
        'mode': 'locked_pc',
        'top_metric': varimax_candidate_df[varimax_candidate_df['pc'] == 'PC2'].iloc[0]['top_metric'],
    },
])

remaining_pc_rows = []
for pc_name in [f'PC{i}' for i in range(3, 9)]:
    spec = CONTROL_SPEC_BY_PC[pc_name]
    raw_row = varimax_candidate_df[varimax_candidate_df['pc'] == pc_name].iloc[0].to_dict()
    remaining_pc_rows.append({
        'pc': pc_name,
        'canonical_name': spec['canonical_name'],
        'control_name': spec['control_name'],
        'control_description': spec['description'],
        'status': spec['status'],
        'top_metric': raw_row['top_metric'],
        'raw_overall_score': raw_row['overall_score'],
        'confidence': raw_row['confidence'],
        'use_recommendation': raw_row['use_recommendation'],
    })
remaining_pc_label_df = pd.DataFrame(remaining_pc_rows).sort_values(
    ['status', 'raw_overall_score'],
    ascending=[True, False],
).reset_index(drop=True)

final_control_registry_rows = []
for _, row in semantic_control_overview_df.iterrows():
    mode = 'locked_pc' if row['control_id'] in {'PC1', 'PC2'} else ('canonical_pc' if row['status'] == 'canonical' else 'unresolved_pc')
    final_control_registry_rows.append({
        'control_id': row['control_id'],
        'canonical_name': row['canonical_name'],
        'control_name': row['control_name'],
        'control_description': row['description'],
        'pc_index': row['pc_index'],
        'mode': mode,
        'semantic_range': row['semantic_range'],
        'pc_range': row['pc_range'],
        'range_source': row['range_source'],
        'top_metric': row['supporting_top_metric'],
        'overall_score': row['supporting_overall_score'],
        'use_recommendation': row['supporting_use_recommendation'],
        'status': row['status'],
        'notes': row['notes'],
    })
final_control_registry_df = pd.DataFrame(final_control_registry_rows)

semantic_control_table_df = final_control_registry_df[[
    'control_id',
    'canonical_name',
    'pc_index',
    'control_name',
    'control_description',
    'semantic_range',
    'pc_range',
    'range_source',
    'top_metric',
    'overall_score',
    'mode',
    'status',
    'notes',
]].copy()

(LOCKED_CONTROL_DIR / 'locked_pc_controls.json').write_text(
    json.dumps(locked_pc_controls, indent=2),
    encoding='utf-8',
)
remaining_pc_label_df.to_csv(LOCKED_CONTROL_DIR / 'remaining_pc_label_summary.csv', index=False)
final_control_registry_df.to_csv(LOCKED_CONTROL_DIR / 'final_control_registry.csv', index=False)
semantic_control_table_df.to_csv(LOCKED_CONTROL_DIR / 'semantic_control_table.csv', index=False)

print('Locked canonical controls:')
display(locked_pc_controls_table)
print('PC3-PC8 canonical / unresolved registry:')
display(remaining_pc_label_df)
print('Semantic control table:')
display(semantic_control_table_df)

In [ ]:
# Cell 12: Conditional sweep with PC1 fixed
DISPLAY_TITLE = '### Cell 12: Conditional sweep with `PC1 = -3.7`'
display(Markdown(DISPLAY_TITLE))

display(Markdown('''This stage fixes the frequency axis before sweeping the remaining controls:

- `PC1 = -3.7`
- sweep `PC2-8`

This lets us inspect how the other axes behave when frequency is held at a fixed operating point.'''))

FOCUSED_SWEEP_DIR = OUTPUT_DIR / 'focused_sweeps'
FOCUSED_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
SHARED_CONDITIONAL_ANCHOR_ID = 0
SHARED_CONDITIONAL_BASE_REFERENCE = varimax_anchor_refs[SHARED_CONDITIONAL_ANCHOR_ID].copy()

shared_conditional_reference_row = {
    'anchor_id': int(SHARED_CONDITIONAL_ANCHOR_ID),
    'row_index': int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID]),
    'sample_id': sample_ids[int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID])],
}
shared_conditional_reference_row.update({
    f'PC{i+1}': float(SHARED_CONDITIONAL_BASE_REFERENCE[i])
    for i in range(len(SHARED_CONDITIONAL_BASE_REFERENCE))
})

print('Shared conditional anchor/reference used by all three sweep blocks:')
display(pd.DataFrame([shared_conditional_reference_row]))

pc_label_lookup = {
    'PC1': {
        'control_family': CONTROL_SPEC_BY_PC['PC1']['canonical_name'],
        'control_name': CONTROL_SPEC_BY_PC['PC1']['control_name'],
        'top_metric': varimax_candidate_df[varimax_candidate_df['pc'] == 'PC1'].iloc[0]['top_metric'],
        'raw_overall_score': None,
        'confidence': 'locked',
        'use_recommendation': 'use',
    },
    'PC2': {
        'control_family': CONTROL_SPEC_BY_PC['PC2']['canonical_name'],
        'control_name': CONTROL_SPEC_BY_PC['PC2']['control_name'],
        'top_metric': varimax_candidate_df[varimax_candidate_df['pc'] == 'PC2'].iloc[0]['top_metric'],
        'raw_overall_score': None,
        'confidence': 'locked',
        'use_recommendation': 'use',
    },
}
for _, row in remaining_pc_label_df.iterrows():
    pc_label_lookup[row['pc']] = {
        'control_family': row['canonical_name'],
        'control_name': row['control_name'],
        'top_metric': row['top_metric'],
        'raw_overall_score': row['raw_overall_score'],
        'confidence': row['confidence'],
        'use_recommendation': row['use_recommendation'],
    }


def run_conditional_pc_sweeps(stage_name, lock_values, pc_names):
    out_dir = FOCUSED_SWEEP_DIR / stage_name
    out_dir.mkdir(parents=True, exist_ok=True)

    reference = SHARED_CONDITIONAL_BASE_REFERENCE.copy()
    for locked_pc, locked_value in lock_values.items():
        axis = int(str(locked_pc).replace('PC', '')) - 1
        reference[axis] = float(locked_value)

    payload = {
        'stage_name': stage_name,
        'anchor_id': int(SHARED_CONDITIONAL_ANCHOR_ID),
        'row_index': int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID]),
        'sample_id': sample_ids[int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID])],
        'shared_base_reference': {f'PC{i+1}': float(SHARED_CONDITIONAL_BASE_REFERENCE[i]) for i in range(len(SHARED_CONDITIONAL_BASE_REFERENCE))},
        'locked_reference': {k: float(v) for k, v in lock_values.items()},
        'selected_pcs': list(pc_names),
        'sweeps': {},
    }
    summary_rows = []

    for pc_name in pc_names:
        meta = pc_label_lookup[pc_name]
        axis = int(pc_name.replace('PC', '')) - 1
        sweep_range = _axis_range(axis, varimax_ranges)
        result = sweep_axis(
            varimax_pipe,
            model,
            DEVICE,
            axis=axis,
            sweep_range=sweep_range,
            n_steps=SINGLE_AXIS_STEPS,
            T=T,
            sr=sr,
            reference=reference,
            with_metrics=True,
        )
        plot_path = out_dir / f'{pc_name}_waveforms.png'
        plot_sweep(result, sr=sr, save_path=str(plot_path), overlay=True)

        summary_rows.append({
            'pc': pc_name,
            'control_family': meta['control_family'],
            'control_name': meta.get('control_name', meta['control_family']),
            'top_metric': meta['top_metric'],
            'raw_overall_score': meta['raw_overall_score'],
            'confidence': meta['confidence'],
            'use_recommendation': meta['use_recommendation'],
            'plot_path': str(plot_path),
        })
        payload['sweeps'][pc_name] = {
            'axis': axis,
            'control_family': meta['control_family'],
            'control_name': meta.get('control_name', meta['control_family']),
            'top_metric': meta['top_metric'],
            'raw_overall_score': meta['raw_overall_score'],
            'values': [float(v) for v in result['values']],
            'metrics': [{k: float(v) for k, v in metric_row.items()} for metric_row in result['metrics']],
            'plot_path': str(plot_path),
        }

    summary_df = pd.DataFrame(summary_rows)
    payload_path = out_dir / f'{stage_name}_sweeps.json'
    summary_path = out_dir / f'{stage_name}_summary.csv'
    payload_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    summary_df.to_csv(summary_path, index=False)

    print('Effective reference for this block:')
    locked_row = {
        'anchor_id': int(SHARED_CONDITIONAL_ANCHOR_ID),
        'row_index': int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID]),
        'sample_id': sample_ids[int(anchor_indices[SHARED_CONDITIONAL_ANCHOR_ID])],
    }
    locked_row.update({f'PC{i+1}': float(reference[i]) for i in range(len(reference))})
    display(pd.DataFrame([locked_row]))
    print('Swept PCs:')
    display(summary_df[['pc', 'control_name', 'control_family', 'top_metric', 'raw_overall_score', 'confidence', 'use_recommendation']])

    return {
        'out_dir': str(out_dir),
        'payload_path': str(payload_path),
        'summary_path': str(summary_path),
        'summary_df': summary_df,
    }

pc1_locked_result = run_conditional_pc_sweeps(
    stage_name='conditional_locked_pc1',
    lock_values={'PC1': -3.7},
    pc_names=[f'PC{i}' for i in range(2, 9)],
)


In [ ]:
# Cell 13: Conditional sweep with PC2 fixed
display(Markdown('### Cell 13: Conditional sweep with `PC2 = -3.0`'))

display(Markdown('''This stage fixes the intensity axis before sweeping the remaining controls:

- `PC2 = -3.0`
- sweep `PC1, PC3-8`

This helps us check which differences remain visible when overall amplitude is held at a fixed operating point.'''))

pc2_locked_result = run_conditional_pc_sweeps(
    stage_name='conditional_locked_pc2',
    lock_values={'PC2': -3.0},
    pc_names=['PC1'] + [f'PC{i}' for i in range(3, 9)],
)


In [ ]:
# Cell 14: Conditional sweep with PC1 and PC2 fixed
display(Markdown('### Cell 14: Conditional sweep with `PC1 = -3.7` and `PC2 = -3.0`'))

display(Markdown('''This stage fixes both primary locked controls before sweeping the remaining PCs:

- `PC1 = -3.7`
- `PC2 = -3.0`
- sweep `PC3-8`

This is the cleanest condition for checking residual control behaviour after frequency and intensity are both pinned.'''))

pc1_pc2_locked_result = run_conditional_pc_sweeps(
    stage_name='conditional_locked_pc1_pc2',
    lock_values={'PC1': -3.7, 'PC2': -3.0},
    pc_names=[f'PC{i}' for i in range(3, 9)],
)


In [ ]:
# Cell 15: Output manifest
display(Markdown('### Cell 15: Output manifest'))

output_manifest = {
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'latent_cache': str(LATENT_PATH),
    'varimax_dir': str(VARIMAX_DIR),
    'varimax_sweep_payload': str(VARIMAX_SWEEP_SUBDIR / 'varimax_multi_anchor_sweeps.json'),
    'varimax_candidate_summary_csv': str(SUMMARY_DIR / 'varimax_candidate_summary.csv'),
    'varimax_candidate_summary_json': str(SUMMARY_DIR / 'varimax_candidate_summary.json'),
    'semantic_control_schema_json': str(SUMMARY_DIR / 'semantic_control_schema.json'),
    'semantic_control_schema_table_csv': str(SUMMARY_DIR / 'semantic_control_table.csv'),
    'semantic_control_overview_csv': str(SUMMARY_DIR / 'semantic_control_overview.csv'),
    'semantic_control_overview_json': str(SUMMARY_DIR / 'semantic_control_overview.json'),
    'locked_pc_controls_json': str(SUMMARY_DIR / 'locked_controls' / 'locked_pc_controls.json'),
    'remaining_pc_label_summary_csv': str(SUMMARY_DIR / 'locked_controls' / 'remaining_pc_label_summary.csv'),
    'final_control_registry_csv': str(SUMMARY_DIR / 'locked_controls' / 'final_control_registry.csv'),
    'semantic_control_table_csv': str(SUMMARY_DIR / 'locked_controls' / 'semantic_control_table.csv'),
    'conditional_locked_pc1_sweeps_json': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc1' / 'conditional_locked_pc1_sweeps.json'),
    'conditional_locked_pc1_summary_csv': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc1' / 'conditional_locked_pc1_summary.csv'),
    'conditional_locked_pc2_sweeps_json': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc2' / 'conditional_locked_pc2_sweeps.json'),
    'conditional_locked_pc2_summary_csv': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc2' / 'conditional_locked_pc2_summary.csv'),
    'conditional_locked_pc1_pc2_sweeps_json': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc1_pc2' / 'conditional_locked_pc1_pc2_sweeps.json'),
    'conditional_locked_pc1_pc2_summary_csv': str(FOCUSED_SWEEP_DIR / 'conditional_locked_pc1_pc2' / 'conditional_locked_pc1_pc2_summary.csv'),
    'figures_dir': str(FIGURE_DIR),
}
output_manifest_path = OUTPUT_DIR / 'varimax_first_output_manifest.json'
output_manifest_path.write_text(json.dumps(output_manifest, indent=2), encoding='utf-8')
print('Saved output manifest:', output_manifest_path)
display(pd.DataFrame([output_manifest]))
